# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nithishreddy08/flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: Prioritize content that shows a strong opportunity for action: high search demand, good CTR potential, and declining or weak performance. Give higher scores to items where an action is likely to improve results. Reason codes:

HIGH_OPPORTUNITY — strong search opportunity and performance gap DECLINING — performance is declining LOW_CTR — CTR is below expectations HIGH_SEARCH_VOLUME — high search demand **LOW_POSITION **— poor search position REVIEW — needs human review before action

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
!git clone https://github.com/nithishreddy08/flyrank.git

Cloning into 'flyrank'...
remote: Enumerating objects: 178, done.
remote: Counting objects: 100% (178/178), done.
remote: Compressing objects: 100% (132/132), done.
remote: Total 178 (delta 82), reused 94 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (178/178), 1.87 MiB | 14.30 MiB/s, done.
Resolving deltas: 100% (82/82), done.


In [3]:
import os

os.chdir("/content/flyrank")

print(os.getcwd())
print(os.path.exists("data/raw/content_refresh_anonymized.csv"))

/content/flyrank
True


In [4]:
import os

print("Current folder:", os.getcwd())

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file == "content_refresh_anonymized.csv":
            print("FOUND:", os.path.join(root, file))

Current folder: /content/flyrank
FOUND: /content/flyrank/data/raw/content_refresh_anonymized.csv


In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load the dataset
input_path = "/content/flyrank/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(input_path)

# Helper: safely convert columns to numbers
def num_col(name):
    if name in df.columns:
        return pd.to_numeric(df[name], errors="coerce")
    return pd.Series(0, index=df.index, dtype=float)

# Important columns
search_volume = num_col("search_volume").fillna(0)
ctr = num_col("ctr").fillna(0)
avg_position = num_col("avg_position").replace(0, np.nan)
trend_pct = num_col("trend_pct").fillna(0)

# Normalize values to 0-1
def minmax(s):
    s = s.fillna(s.median() if s.notna().any() else 0)
    if s.max() == s.min():
        return pd.Series(0, index=s.index)
    return (s - s.min()) / (s.max() - s.min())

search_score = minmax(search_volume)
ctr_score = 1 - minmax(ctr)              # lower CTR = more opportunity
position_score = minmax(avg_position)    # higher position number = worse position
decline_score = minmax((-trend_pct).clip(lower=0))

# Baseline action score
df["baseline_action_score"] = (
    0.35 * search_score
    + 0.25 * ctr_score
    + 0.20 * position_score
       + 0.20 * decline_score
)

# Add reason codes
def reason_codes(row):
    reasons = []

    if row["search_volume"] >= search_volume.quantile(0.75):
        reasons.append("HIGH_SEARCH_VOLUME")

    if row["ctr"] <= ctr.quantile(0.25):
        reasons.append("LOW_CTR")

    if pd.notna(row["avg_position"]) and row["avg_position"] >= avg_position.quantile(0.75):
        reasons.append("LOW_POSITION")

    if row["trend_pct"] < 0:
        reasons.append("DECLINING")

    if len(reasons) == 0:
        reasons.append("REVIEW")

    return "|".join(reasons)

df["reason_codes"] = df.apply(reason_codes, axis=1)

# Rank all content
df = df.sort_values(
    "baseline_action_score",
    ascending=False
).reset_index(drop=True)

df["rank"] = df.index + 1
# Create output directory
from pathlib import Path
Path("work/outputs").mkdir(parents=True, exist_ok=True)

# Save ranked queue - Quotes close chey + space teesi _ pettu
output_path = "work/outputs/baseline_action_score.csv"  # <-- ikkada fix
df.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Rows ranked: {len(df)}")
print("\nTop 20:")
display(
    df[["rank", "baseline_action_score", "reason_codes"]].head(20)
)



Saved: work/outputs/baseline_action_score.csv
Rows ranked: 30000

Top 20:


,rank,baseline_action_score,reason_codes
0,1,0.678335,HIGH_SEARCH_VOLUME|LOW_CTR|LOW_POSITION|DECLINING
1,2,0.652668,HIGH_SEARCH_VOLUME|LOW_CTR|LOW_POSITION|DECLINING
2,3,0.631285,HIGH_SEARCH_VOLUME|LOW_POSITION
3,4,0.613225,HIGH_SEARCH_VOLUME|LOW_CTR|LOW_POSITION|DECLINING
4,5,0.591977,HIGH_SEARCH_VOLUME|LOW_CTR|DECLINING
5,6,0.576737,HIGH_SEARCH_VOLUME|LOW_CTR|LOW_POSITION
6,7,0.575122,HIGH_SEARCH_VOLUME|LOW_CTR|LOW_POSITION|DECLINING
7,8,0.567223,HIGH_SEARCH_VOLUME|DECLINING
8,9,0.563220,HIGH_SEARCH_VOLUME|LOW_CTR|LOW_POSITION|DECLINING
9,10,0.546473,HIGH_SEARCH_VOLUME|LOW_CTR|LOW_POSITION|DECLINING


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
top20 = df.head(20).copy()

# Create action if it does not already exist
def get_action(row):
    if row["baseline_action_score"] >= 0.75:
        return "refresh_priority"
    elif row["baseline_action_score"] >= 0.50:
        return "review"
    else:
        return "monitor"

top20["action"] = top20.apply(get_action, axis=1)

# Confidence note
def confidence_note(row):
    if row["baseline_action_score"] >= 0.75:
        return "Higher directional confidence."
    elif row["baseline_action_score"] >= 0.50:
        return "Medium directional confidence; human review needed."
    else:
        return "Lower directional confidence."

top20["confidence_note"] = top20.apply(confidence_note, axis=1)

# What could make it wrong
top20["what_would_make_it_wrong"] = (
    "Could be wrong if important business or page context is missing."
)

# Select columns that definitely exist
top20_review = top20[
    [
        "rank",
        "content_id",
        "baseline_action_score",
        "action",
        "reason_codes",
                "confidence_note",
        "what_would_make_it_wrong"
    ]
]

display(top20_review)

# Save Top-20 review
top20_review.to_csv(
    "work/outputs/top20_review.csv",
    index=False
)

print("Saved: work/outputs/top20_review.csv")

,rank,content_id,baseline_action_score,action,reason_codes,confidence_note,what_would_make_it_wrong
0,1,content_454cc6654c6e,0.678335,review,HIGH_SEARCH_VOLUME|LOW_CTR|LOW_POSITION|DECLINING,Medium directional confidence; human review ne...,Could be wrong if important business or page c...
1,2,content_cd6760921db8,0.652668,review,HIGH_SEARCH_VOLUME|LOW_CTR|LOW_POSITION|DECLINING,Medium directional confidence; human review ne...,Could be wrong if important business or page c...
2,3,content_ef99c4abd9ab,0.631285,review,HIGH_SEARCH_VOLUME|LOW_POSITION,Medium directional confidence; human review ne...,Could be wrong if important business or page c...
3,4,content_bf67a444faef,0.613225,review,HIGH_SEARCH_VOLUME|LOW_CTR|LOW_POSITION|DECLINING,Medium directional confidence; human review ne...,Could be wrong if important business or page c...
4,5,content_12e48d4b449d,0.591977,review,HIGH_SEARCH_VOLUME|LOW_CTR|DECLINING,Medium directional confidence; human review ne...,Could be wrong if important business or page c...
5,6,content_5ec29ae79c60,0.576737,review,HIGH_SEARCH_VOLUME|LOW_CTR|LOW_POSITION,Medium directional confidence; human review ne...,Could be wrong if important business or page c...
6,7,content_deb54e9e19cd,0.575122,review,HIGH_SEARCH_VOLUME|LOW_CTR|LOW_POSITION|DECLINING,Medium directional confidence; human review ne...,Could be wrong if important business or page c...
7,8,content_f76ccf7a7834,0.567223,review,HIGH_SEARCH_VOLUME|DECLINING,Medium directional confidence; human review ne...,Could be wrong if important business or page c...
8,9,content_ee4630879d03,0.563220,review,HIGH_SEARCH_VOLUME|LOW_CTR|LOW_POSITION|DECLINING,Medium directional confidence; human review ne...,Could be wrong if important business or page c...
9,10,content_13bbd72aea33,0.546473,review,HIGH_SEARCH_VOLUME|LOW_CTR|LOW_POSITION|DECLINING,Medium directional confidence; human review ne...,Could be wrong if important business or page c...


Saved: work/outputs/top20_review.csv


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [8]:
print("=== Weak Picks ===")

# Top 20 weaker picks based on score
weak_picks = df.head(20).tail(5)[
    [
        "rank",
        "content_id",
        "baseline_action_score",
        "reason_codes"
    ]
]

display(weak_picks)


# =========================
# Leakage Check
# =========================

print("\n=== Leakage Check ===")

# Fields that must NOT be used in the score
forbidden_features = [
    "trend_pct",
    "trend_direction",
    "is_declining_label",
    "client_id",
    "content_id"
]

# Fields actually used by our baseline score
score_features = [
    "impressions_90d",
    "days_since_last_update",
        "avg_position",
    "word_count"
]

print("Score features used:")
print(score_features)

print("\nForbidden fields checked:")
print(forbidden_features)


# Check that forbidden fields are not score inputs
score_formula_text = """
impressions_90d
days_since_last_update
avg_position
word_count
"""


=== Weak Picks ===


,rank,content_id,baseline_action_score,reason_codes
15,16,content_638236e8066e,0.530613,HIGH_SEARCH_VOLUME|LOW_CTR|LOW_POSITION|DECLINING
16,17,content_f616ca0ec5ea,0.527488,HIGH_SEARCH_VOLUME|LOW_CTR|LOW_POSITION|DECLINING
17,18,content_3f7dbbd55f0c,0.524848,HIGH_SEARCH_VOLUME|LOW_CTR|DECLINING
18,19,content_6b41450ae50c,0.524374,HIGH_SEARCH_VOLUME|LOW_CTR|LOW_POSITION|DECLINING
19,20,content_86748254b6bf,0.523418,LOW_CTR|LOW_POSITION|DECLINING



=== Leakage Check ===
Score features used:
['impressions_90d', 'days_since_last_update', 'avg_position', 'word_count']

Forbidden fields checked:
['trend_pct', 'trend_direction', 'is_declining_label', 'client_id', 'content_id']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.